# FT-Transformer — Employee Resignation Prediction

**Member 1 model:** FT-Transformer  
**Task:** Binary classification  
**Target:** `Resigned` (`True` / `1` = resigned)  
**Primary metric:** Average Precision (reported as PR-AUC in project discussion)

## Objective

Train and evaluate FT-Transformer on the project's **80,000-row development partition** using the frozen **5-fold cross-validation** registry and all **17 approved predictors**.

The **20,000-row formal holdout is excluded** from training, preprocessing, tuning, model selection, and OOF evaluation. It remains reserved for the locked final evaluation stage.

## Methodology and Leakage Controls

Each outer CV round uses **64,000 training rows** and **16,000 validation rows**.

The FT-Transformer adapter owns fold-local preprocessing:

- 13 numerical/ordinal predictors are standardized with `StandardScaler` fitted on the outer-training folds only.
- 4 categorical predictors are encoded from mappings learned on the outer-training folds only.
- A fresh adapter is created for each outer fold.
- The outer validation fold is never passed to `fit()`.
- One probability is generated for every development employee, producing 80,000 OOF predictions.
- `Employee_ID`, `Hire_Date`, and `Resigned` are never used as predictors.

The shared repository runner also rejects any formal-holdout `Employee_ID` passed into OOF training.

In [ ]:
# -----------------------------
# 1. Imports and project setup
# -----------------------------

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from sklearn.metrics import precision_recall_curve

# Resolve the repository root from notebooks/05_member_1/.
PROJECT_ROOT = Path("../..").resolve()

# Fallback for notebook launch from a different working directory.
if not (PROJECT_ROOT / "configs" / "feature_contract.yaml").exists():
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "configs" / "feature_contract.yaml").exists():
            PROJECT_ROOT = candidate.resolve()
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import calculate_binary_metrics
from src.models.ft_transformer import FTTransformerAdapter
from src.models.runner import run_5fold_cv

pd.set_option("display.max_columns", None)

CONFIG_PATH = PROJECT_ROOT / "configs" / "models" / "ft_transformer.yaml"
FEATURE_CONTRACT_PATH = PROJECT_ROOT / "configs" / "feature_contract.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "models" / "ft_transformer"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [CONFIG_PATH, FEATURE_CONTRACT_PATH, DATA_PATH, SPLIT_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required project files: {missing_paths}"

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# -----------------------------
# 2. Load configuration and record the executed parameter set
# -----------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)

with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)

model_name = ft_config["model"]["name"]
model_id = ft_config["model"]["model_id"]

# Start from the fixed architecture and make optimizer defaults explicit.
fixed_params = ft_config["fixed_params"].copy()
fixed_params.setdefault(
    "learning_rate",
    float(ft_config["search_space"]["learning_rate"]["default"]),
)
fixed_params.setdefault(
    "weight_decay",
    float(ft_config["search_space"]["weight_decay"]["default"]),
)

config_summary = pd.DataFrame(
    [
        ("Model", model_name),
        ("Model ID", model_id),
        ("Implementation", ft_config["model"]["implementation"]),
        ("Seed", fixed_params["seed"]),
        ("Transformer blocks", fixed_params["n_blocks"]),
        ("Block dimension", fixed_params["d_block"]),
        ("Attention heads", fixed_params["attention_n_heads"]),
        ("FFN hidden dimension", fixed_params["ffn_d_hidden"]),
        ("Learning rate", fixed_params["learning_rate"]),
        ("Weight decay", fixed_params["weight_decay"]),
        ("Executed epochs per fold", 50),
        ("Batch size", 512),
    ],
    columns=["Setting", "Value"],
)

display(config_summary)

print("\nConfigured search-space parameters:")
display(
    pd.DataFrame(ft_config["search_space"])
    .T
    .rename_axis("parameter")
    .reset_index()
)

In [ ]:
# -----------------------------
# 3. Reproducibility and compute environment
# -----------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_label = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Python kernel:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Training device:", device_label)

if torch.cuda.is_available():
    print("PyTorch CUDA runtime:", torch.version.cuda)
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("Random seed:", fixed_params["seed"])

In [ ]:
# -----------------------------
# 4. Load raw data and frozen split registry
# -----------------------------

df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

assert df.shape == (100_000, 20), f"Unexpected raw dataset shape: {df.shape}"
assert split_registry.shape == (100_000, 3), (
    f"Unexpected split-registry shape: {split_registry.shape}"
)
assert df["Employee_ID"].is_unique
assert split_registry["Employee_ID"].is_unique

print("Raw dataset shape:", df.shape)
print("Split registry shape:", split_registry.shape)
print("\nPartition counts:")
print(split_registry["partition"].value_counts())

In [ ]:
# -----------------------------
# 5. Isolate the development partition and enforce holdout protection
# -----------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID",
    ]
)
holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID",
    ]
)

development_data = (
    df.loc[df["Employee_ID"].isin(development_ids)]
    .copy()
    .reset_index(drop=True)
)

development_df = development_data.merge(
    split_registry[["Employee_ID", "partition", "cv_fold"]],
    on="Employee_ID",
    how="left",
    validate="one_to_one",
)
development_df["cv_fold"] = development_df["cv_fold"].astype(int)

assert len(development_data) == 80_000
assert len(holdout_ids) == 20_000
assert set(development_data["Employee_ID"]).isdisjoint(holdout_ids)
assert set(development_df["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert development_df["cv_fold"].value_counts().eq(16_000).all()

print("Development rows:", len(development_data))
print("Formal holdout rows excluded:", len(holdout_ids))
print("Holdout IDs present in modeling data: 0")
print("\nDevelopment fold counts:")
print(development_df["cv_fold"].value_counts().sort_index())

In [ ]:
# -----------------------------
# 6. Validate the feature contract and build X / y
# -----------------------------

numerical_features = feature_contract["numerical_features"]
categorical_features = feature_contract["categorical_features"]
primary_model_features = feature_contract["primary_model_features"]
target_config = feature_contract["target"]

numerical_feature_names = [item["name"] for item in numerical_features]
categorical_feature_names = [item["name"] for item in categorical_features]
target_column = target_config["column"]

assert len(numerical_feature_names) == 13
assert len(categorical_feature_names) == 4
assert len(primary_model_features) == 17
assert set(numerical_feature_names + categorical_feature_names) == set(primary_model_features)

required_columns = primary_model_features + [target_column, "Employee_ID", "cv_fold"]
missing_columns = [c for c in required_columns if c not in development_df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

X = development_df[primary_model_features].copy()
y = development_df[target_column].astype(int).copy()
employee_ids = development_df["Employee_ID"].copy()
cv_folds = development_df["cv_fold"].copy()

assert X.shape == (80_000, 17)
assert len(y) == 80_000
assert set(y.unique()) == {0, 1}
assert X.isna().sum().sum() == 0
assert "Employee_ID" not in X.columns
assert "Hire_Date" not in X.columns
assert target_column not in X.columns

print("Predictor matrix:", X.shape)
print("Numerical / ordinal predictors:", len(numerical_feature_names))
print("Categorical predictors:", len(categorical_feature_names))
print("Missing predictor values:", int(X.isna().sum().sum()))
print("Development resignation prevalence:", f"{y.mean():.4%}")

In [ ]:
# -----------------------------
# 7. Verify fold balance
# -----------------------------

fold_summary = (
    development_df
    .groupby("cv_fold")[target_column]
    .agg(
        total_rows="size",
        resigned_count="sum",
        resignation_rate="mean",
    )
    .reset_index()
)

fold_summary["not_resigned_count"] = (
    fold_summary["total_rows"] - fold_summary["resigned_count"]
)
fold_summary["resignation_rate_pct"] = (
    fold_summary["resignation_rate"] * 100
).round(3)

fold_summary = fold_summary[
    [
        "cv_fold",
        "total_rows",
        "not_resigned_count",
        "resigned_count",
        "resignation_rate_pct",
    ]
]

assert len(fold_summary) == 5
assert (fold_summary["total_rows"] == 16_000).all()
assert (fold_summary["resigned_count"] > 0).all()
assert (fold_summary["not_resigned_count"] > 0).all()

display(fold_summary)

In [ ]:
# -----------------------------
# 8. Verify categorical coverage across outer folds
# -----------------------------

coverage_records = []

for fold in sorted(development_df["cv_fold"].unique()):
    train_fold_df = development_df[development_df["cv_fold"] != fold]
    val_fold_df = development_df[development_df["cv_fold"] == fold]

    for feature in categorical_feature_names:
        train_categories = set(train_fold_df[feature].dropna().unique())
        validation_categories = set(val_fold_df[feature].dropna().unique())
        unseen = validation_categories - train_categories

        coverage_records.append(
            {
                "cv_fold": fold,
                "feature": feature,
                "train_categories": len(train_categories),
                "validation_categories": len(validation_categories),
                "unseen_categories": len(unseen),
            }
        )

categorical_coverage = pd.DataFrame(coverage_records)
total_unseen = int(categorical_coverage["unseen_categories"].sum())

assert total_unseen == 0, (
    "At least one outer-validation fold contains a category absent "
    "from its corresponding training folds."
)

display(categorical_coverage)
print("\nTotal unseen validation categories:", total_unseen)

## Verified FT-Transformer Execution Protocol

The repository implementation was inspected before training.

**Executed behavior**

- Fresh `FTTransformerAdapter` per outer fold.
- Fold-local `StandardScaler` and categorical mappings.
- AdamW optimizer.
- `BCEWithLogitsLoss`.
- 50 epochs per fold.
- Batch size 512.
- Learning rate 0.001.
- Weight decay 1e-5.
- Seed 42.
- Automatic device selection (`cuda` when available, otherwise CPU).

**Configuration caveat**

The YAML also declares `max_epochs: 100`, `patience: 15`, `ReduceLROnPlateau`, and an inner validation fraction of 0.15. The current shared runner calls `adapter.fit(X, y)` without supplying those settings, and the current FT-Transformer adapter does not implement inner-validation early stopping or the scheduler. This notebook therefore reports the behavior that is **actually executed**, not the unused YAML declarations.

The YAML search space is recorded for reproducibility, but this completed run uses the fixed/default configuration; it is **not presented as a completed hyperparameter-search result**.

The source adapter also passes `ffn_d_hidden_multiplier=None` for compatibility with `rtdl_revisiting_models==0.0.2` while using the explicit `ffn_d_hidden` value.

In [ ]:
# -----------------------------
# 9. Create a fresh adapter factory
# -----------------------------

def make_ft_transformer():
    """Return a fresh, unfitted FT-Transformer adapter."""
    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )

adapter_a = make_ft_transformer()
adapter_b = make_ft_transformer()

assert adapter_a is not adapter_b
assert adapter_a.params is not adapter_b.params
assert adapter_a.categorical_features is not adapter_b.categorical_features
assert not adapter_a.get_metadata()["fitted"]
assert not adapter_b.get_metadata()["fitted"]

print("Adapter factory validation passed.")
print("Model:", adapter_a.name)
print("Categorical predictors:", adapter_a.categorical_features)

In [ ]:
# -----------------------------
# 10. Official 5-fold OOF training
# -----------------------------

print("FT-Transformer official CV training")
print("-----------------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer CV folds: 5")
print("Training rows per fold: 64,000")
print("Validation rows per fold: 16,000")
print("Epochs per fold: 50")
print("Batch size: 512")
print("Training device:", device_label)

cv_start_time = time.perf_counter()

ft_cv_result = run_5fold_cv(
    adapter_factory=make_ft_transformer,
    data=development_data,
    split_registry=split_registry,
    feature_cols=primary_model_features,
    target_col=target_column,
    id_col="Employee_ID",
    fold_col="cv_fold",
    partition_col="partition",
    development_label="development",
    holdout_label="holdout",
    expected_folds=(1, 2, 3, 4, 5),
    expected_model_name=model_name,
)

cv_elapsed_time = time.perf_counter() - cv_start_time

print("\nFT-Transformer 5-fold CV completed successfully.")
print("Wall-clock time:", f"{cv_elapsed_time / 60:.2f} minutes")
print("OOF prediction rows:", len(ft_cv_result.predictions))

### Recorded Successful Run

The uploaded working notebook already completed the official 5-fold development run successfully. The recorded result was:

| Metric | Recorded value |
|---|---:|
| Pooled Average Precision / PR-AUC | **0.101132** |
| Pooled ROC-AUC | **0.498821** |
| Pooled Brier score | **0.090651** |
| Pooled F1 @ 0.50 | **0.000000** |
| CV mean Average Precision | **0.102028** |
| CV SD Average Precision | **0.002378** |
| CV mean ROC-AUC | **0.499769** |
| CV SD ROC-AUC | **0.010585** |
| Total fit time | **770.90 s** |
| Total prediction time | **1.16 s** |
| Wall-clock time reported by notebook | **12.88 min** |

CUDA was verified as available with an **NVIDIA GeForce RTX 4060 Laptop GPU (8 GB)** before this run. The earlier `"Training device: CPU"` text in the development notebook was a hard-coded print statement; the adapter itself selects CUDA dynamically when `torch.cuda.is_available()` is `True`.

In [ ]:
# -----------------------------
# 11. Fold-level performance
# -----------------------------

fold_metrics = ft_cv_result.fold_metrics.copy()

expected_metric_columns = {
    "fold",
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
    "brier_score",
    "fit_time_seconds",
    "predict_time_seconds",
}
missing_metric_columns = expected_metric_columns - set(fold_metrics.columns)
assert not missing_metric_columns, (
    f"Fold metrics are missing columns: {sorted(missing_metric_columns)}"
)
assert len(fold_metrics) == 5

display(
    fold_metrics[
        [
            "fold",
            "average_precision",
            "roc_auc",
            "brier_score",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "fit_time_seconds",
            "predict_time_seconds",
        ]
    ].round(6)
)

In [ ]:
# -----------------------------
# 12. Aggregate OOF metrics and no-skill reference
# -----------------------------

oof_predictions = ft_cv_result.predictions.copy()

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].is_unique
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert set(oof_predictions["cv_fold"].unique()) == {1, 2, 3, 4, 5}
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)

pooled_metrics = calculate_binary_metrics(
    y_true=oof_predictions["y_true"],
    y_probability=oof_predictions["y_probability"],
    y_pred=oof_predictions["y_pred_0_5"],
    threshold=0.50,
)

prevalence = float(oof_predictions["y_true"].mean())
no_skill_ap = prevalence
constant_prevalence_brier = prevalence * (1.0 - prevalence)

metric_summary = pd.DataFrame(
    [
        ("Average Precision / PR-AUC", pooled_metrics["average_precision"]),
        ("No-skill AP (prevalence)", no_skill_ap),
        ("ROC-AUC", pooled_metrics["roc_auc"]),
        ("Brier score", pooled_metrics["brier_score"]),
        ("Constant-prevalence Brier reference", constant_prevalence_brier),
        ("Balanced accuracy @ 0.50", pooled_metrics["balanced_accuracy"]),
        ("Precision @ 0.50", pooled_metrics["precision"]),
        ("Recall @ 0.50", pooled_metrics["recall"]),
        ("F1 @ 0.50", pooled_metrics["f1"]),
        ("Predicted-positive rate @ 0.50", pooled_metrics["predicted_positive_rate"]),
    ],
    columns=["Metric", "Value"],
)

display(metric_summary.round(6))

print(
    "\nPR-AUC lift over prevalence:",
    f"{pooled_metrics['average_precision'] - no_skill_ap:+.6f}",
)

In [ ]:
# -----------------------------
# 13. OOF probability diagnostics
# -----------------------------

probabilities = oof_predictions["y_probability"]

probability_summary = pd.DataFrame(
    {
        "Statistic": [
            "Minimum",
            "1st percentile",
            "25th percentile",
            "Median",
            "Mean",
            "75th percentile",
            "99th percentile",
            "Maximum",
        ],
        "Probability": [
            probabilities.min(),
            probabilities.quantile(0.01),
            probabilities.quantile(0.25),
            probabilities.median(),
            probabilities.mean(),
            probabilities.quantile(0.75),
            probabilities.quantile(0.99),
            probabilities.max(),
        ],
    }
)

display(probability_summary.round(6))

print(
    "Predicted positives at threshold 0.50:",
    int(oof_predictions["y_pred_0_5"].sum()),
)
print(
    "Actual positives:",
    int(oof_predictions["y_true"].sum()),
)

In [ ]:
# -----------------------------
# 14. Precision-recall curve and probability distribution
# -----------------------------

precision, recall, _ = precision_recall_curve(
    oof_predictions["y_true"],
    oof_predictions["y_probability"],
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, label=f"FT-Transformer AP = {pooled_metrics['average_precision']:.4f}")
ax.axhline(prevalence, linestyle="--", label=f"No-skill prevalence = {prevalence:.4f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("FT-Transformer OOF Precision-Recall Curve")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

pr_curve_path = RESULTS_DIR / "figure_01_ft_transformer_precision_recall_curve.png"
fig.savefig(pr_curve_path, dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    [
        oof_predictions.loc[oof_predictions["y_true"] == 0, "y_probability"],
        oof_predictions.loc[oof_predictions["y_true"] == 1, "y_probability"],
    ],
    bins=40,
    label=["Not resigned", "Resigned"],
    alpha=0.7,
)
ax.set_xlabel("Predicted resignation probability")
ax.set_ylabel("Employees")
ax.set_title("FT-Transformer OOF Probability Distribution")
ax.legend()
fig.tight_layout()

probability_plot_path = RESULTS_DIR / "figure_02_ft_transformer_oof_probability_distribution.png"
fig.savefig(probability_plot_path, dpi=150, bbox_inches="tight")
plt.show()

## Tuning, Interpretability, and Serialization Scope

### Hyperparameter tuning
The YAML contains a search space, but the recorded run above evaluates the fixed/default configuration. It should **not** be described as a tuned optimum. Any later tuning must remain development-only and reuse the frozen folds without accessing the formal holdout.

### Interpretability
The current FT-Transformer adapter does not expose stable per-feature importance, attention attribution, SHAP values, or retained fold models through the shared OOF runner. This notebook therefore does **not fabricate a feature-importance ranking**. Cross-model interpretability and fairness analyses should use methods that are explicitly supported by the selected model implementation.

### Model serialization
The shared OOF runner returns predictions, metrics, timing, and metadata, but it does not retain all five fitted fold adapters. This notebook therefore serializes the **evaluation artifacts** needed for model comparison. Final deployable-model training and model-weight serialization should occur only after model selection and pipeline freeze.

In [ ]:
# -----------------------------
# 15. Save reproducible FT-Transformer evaluation artifacts
# -----------------------------

predictions_path = RESULTS_DIR / "ft_transformer_oof_predictions.csv"
fold_metrics_path = RESULTS_DIR / "ft_transformer_fold_metrics.csv"
summary_metrics_path = RESULTS_DIR / "ft_transformer_summary_metrics.json"
run_metadata_path = RESULTS_DIR / "ft_transformer_run_metadata.json"
manifest_path = RESULTS_DIR / "ft_transformer_output_manifest.csv"

oof_predictions.to_csv(predictions_path, index=False)
fold_metrics.to_csv(fold_metrics_path, index=False)

with open(summary_metrics_path, "w", encoding="utf-8") as file:
    json.dump(ft_cv_result.summary_metrics, file, indent=2)

run_metadata = {
    "model_name": model_name,
    "model_id": model_id,
    "feature_count": len(primary_model_features),
    "features": primary_model_features,
    "categorical_features": categorical_feature_names,
    "numerical_features": numerical_feature_names,
    "development_rows": len(development_data),
    "formal_holdout_rows_used": 0,
    "outer_folds": [1, 2, 3, 4, 5],
    "epochs_per_fold": 50,
    "batch_size": 512,
    "device": device_label,
    "torch_version": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "executed_params": fixed_params,
    "tuning_performed": False,
    "adapter_metadata": ft_cv_result.metadata,
    "fit_time_seconds_total": ft_cv_result.fit_time_seconds_total,
    "predict_time_seconds_total": ft_cv_result.predict_time_seconds_total,
}

with open(run_metadata_path, "w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

artifact_paths = [
    predictions_path,
    fold_metrics_path,
    summary_metrics_path,
    run_metadata_path,
    pr_curve_path,
    probability_plot_path,
]

manifest = pd.DataFrame(
    {
        "artifact": [path.name for path in artifact_paths],
        "relative_path": [
            str(path.relative_to(PROJECT_ROOT))
            for path in artifact_paths
        ],
        "exists": [path.exists() for path in artifact_paths],
    }
)
manifest.to_csv(manifest_path, index=False)

display(manifest)
print("\nArtifacts saved to:", RESULTS_DIR)

In [ ]:
# -----------------------------
# 16. Final integrity checks
# -----------------------------

assert len(oof_predictions) == 80_000
assert oof_predictions["Employee_ID"].nunique() == 80_000
assert set(oof_predictions["Employee_ID"]).isdisjoint(holdout_ids)
assert oof_predictions["y_probability"].between(0, 1).all()
assert oof_predictions["y_probability"].notna().all()
assert len(fold_metrics) == 5
assert set(fold_metrics["fold"]) == {1, 2, 3, 4, 5}

assert np.isclose(
    pooled_metrics["average_precision"],
    ft_cv_result.summary_metrics["pooled_average_precision"],
)
assert np.isclose(
    pooled_metrics["roc_auc"],
    ft_cv_result.summary_metrics["pooled_roc_auc"],
)
assert np.isclose(
    pooled_metrics["brier_score"],
    ft_cv_result.summary_metrics["pooled_brier_score"],
)

required_artifacts = artifact_paths + [manifest_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
assert not missing_artifacts, f"Missing result artifacts: {missing_artifacts}"

print("All FT-Transformer final integrity checks passed.")
print("Development OOF evaluation is ready for model comparison.")
print("Formal holdout remains untouched.")

## Conclusion

FT-Transformer was evaluated on all **17 approved predictors** using the frozen five-fold development protocol. The formal 20,000-row holdout was not used.

The recorded completed run achieved:

- **Pooled PR-AUC / Average Precision:** 0.101132
- **Pooled ROC-AUC:** 0.498821
- **Pooled Brier score:** 0.090651
- **Pooled F1 at threshold 0.50:** 0.000000

Development-set resignation prevalence is approximately **10.01%**, so the model's PR-AUC is essentially at the no-skill prevalence reference. ROC-AUC is also approximately 0.50, indicating little to no useful ranking discrimination under this configuration. The zero F1 at 0.50 is consistent with probabilities remaining below the default classification threshold.

This is a valid negative modeling result: the current FT-Transformer configuration does not demonstrate meaningful predictive signal on the approved feature set. The OOF artifacts should proceed to the common model-comparison stage, while the formal holdout remains locked.

**Status:** FT-Transformer development OOF evaluation complete; ready for cross-model comparison. Hyperparameter tuning is not claimed as complete.

In [1]:
# -----------------------------
# Weighted FT-Transformer smoke test
# -----------------------------

import importlib
import numpy as np

import src.models.ft_transformer as ft_module


# Reload the updated .py file into the current notebook kernel.
ft_module = importlib.reload(ft_module)

FTTransformerAdapter = ft_module.FTTransformerAdapter


# Take a small development-only sample.
smoke_df = (
    development_df
    .sample(
        n=2048,
        random_state=42
    )
    .reset_index(drop=True)
)

X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = (
    smoke_df[target_column]
    .astype(int)
    .copy()
)


# Create the UPDATED weighted adapter.
weighted_smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params.copy(),
    categorical_features=categorical_feature_names.copy(),
)


# Train for only 1 epoch.
# This is NOT part of the official experiment.
weighted_smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=1,
    batch_size=256,
)


# Check the calculated fold-local positive-class weight.
metadata = weighted_smoke_adapter.get_metadata()

print("Positive samples:", int(y_smoke.sum()))
print("Negative samples:", int((y_smoke == 0).sum()))

print(
    "Calculated pos_weight:",
    metadata["pos_weight"]
)

print(
    "Class weighting:",
    metadata["class_weighting"]
)


# Produce a few predictions to confirm inference works.
smoke_probs = weighted_smoke_adapter.predict_proba(
    X_smoke.iloc[:64]
)

assert metadata["pos_weight"] is not None
assert metadata["pos_weight"] > 1
assert len(smoke_probs) == 64
assert np.isfinite(smoke_probs).all()
assert ((smoke_probs >= 0) & (smoke_probs <= 1)).all()


print("\nWeighted FT-Transformer smoke test PASSED.")

print(
    "Probability range:",
    f"{smoke_probs.min():.6f}",
    "to",
    f"{smoke_probs.max():.6f}"
)

ModuleNotFoundError: No module named 'src'

In [2]:
# -----------------------------
# Restore project imports
# -----------------------------

import sys
from pathlib import Path


# Find the repository root.
PROJECT_ROOT = Path("../..").resolve()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate.resolve()
            break


# Add project root so `src...` imports work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: D:\Desktop\Employee_Resignation_Prediction
src exists: True


In [3]:
# -----------------------------
# Restore variables for weighted FT-Transformer test
# -----------------------------

import yaml
import numpy as np
import pandas as pd
import torch
import importlib

import src.models.ft_transformer as ft_module


# Reload the UPDATED adapter file.
ft_module = importlib.reload(ft_module)
FTTransformerAdapter = ft_module.FTTransformerAdapter


# -----------------------------
# Paths
# -----------------------------

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "models"
    / "ft_transformer.yaml"
)

FEATURE_CONTRACT_PATH = (
    PROJECT_ROOT
    / "configs"
    / "feature_contract.yaml"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Extended_Employee_Performance_and_Productivity_Data.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_registry.csv"
)


# -----------------------------
# Load configuration
# -----------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    ft_config = yaml.safe_load(file)

with open(FEATURE_CONTRACT_PATH, "r", encoding="utf-8") as file:
    feature_contract = yaml.safe_load(file)


# -----------------------------
# Load data
# -----------------------------

df = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

data = df.merge(
    split_registry,
    on="Employee_ID",
    how="left",
    validate="one_to_one",
)

development_df = (
    data.loc[
        data["partition"] == "development"
    ]
    .copy()
    .reset_index(drop=True)
)

development_df["cv_fold"] = (
    development_df["cv_fold"]
    .astype(int)
)


# -----------------------------
# Feature/model definitions
# -----------------------------

primary_model_features = (
    feature_contract["primary_model_features"]
)

categorical_feature_names = [
    item["name"]
    for item in feature_contract["categorical_features"]
]

target_column = (
    feature_contract["target"]["column"]
)

model_name = (
    ft_config["model"]["name"]
)

model_id = (
    ft_config["model"]["model_id"]
)

fixed_params = (
    ft_config["fixed_params"].copy()
)


# -----------------------------
# Quick checks
# -----------------------------

assert len(development_df) == 80_000
assert len(primary_model_features) == 17
assert len(categorical_feature_names) == 4


print("Setup restored successfully.")
print("Development rows:", len(development_df))
print("Predictors:", len(primary_model_features))
print("Categorical features:", len(categorical_feature_names))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Setup restored successfully.
Development rows: 80000
Predictors: 17
Categorical features: 4
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
# -----------------------------
# Weighted FT-Transformer smoke test
# -----------------------------

# Take a small development-only sample.
# This is only a technical test, not the final experiment.
smoke_df = (
    development_df
    .sample(
        n=2048,
        random_state=42
    )
    .reset_index(drop=True)
)


X_smoke = smoke_df[
    primary_model_features
].copy()

y_smoke = (
    smoke_df[target_column]
    .astype(int)
    .copy()
)


# Create the UPDATED weighted adapter.
weighted_smoke_adapter = FTTransformerAdapter(
    name=model_name,
    model_id=model_id,
    params=fixed_params.copy(),
    categorical_features=categorical_feature_names.copy(),
)


# Train for only 1 epoch.
weighted_smoke_adapter.fit(
    X=X_smoke,
    y=y_smoke,
    epochs=1,
    batch_size=256,
)


# Read metadata after training.
metadata = weighted_smoke_adapter.get_metadata()


# -----------------------------
# Inspect class weighting
# -----------------------------

positive_count = int(
    y_smoke.sum()
)

negative_count = int(
    (y_smoke == 0).sum()
)


print("Positive samples:", positive_count)
print("Negative samples:", negative_count)

print(
    "Expected pos_weight:",
    negative_count / positive_count
)

print(
    "Adapter pos_weight:",
    metadata["pos_weight"]
)

print(
    "Class weighting:",
    metadata["class_weighting"]
)


# -----------------------------
# Test prediction
# -----------------------------

smoke_probs = (
    weighted_smoke_adapter
    .predict_proba(
        X_smoke.iloc[:64]
    )
)


# -----------------------------
# Validation checks
# -----------------------------

assert metadata["pos_weight"] is not None

assert metadata["pos_weight"] > 1

assert np.isclose(
    metadata["pos_weight"],
    negative_count / positive_count
)

assert len(smoke_probs) == 64

assert np.isfinite(
    smoke_probs
).all()

assert (
    (smoke_probs >= 0)
    & (smoke_probs <= 1)
).all()


print(
    "\nWeighted FT-Transformer smoke test PASSED."
)

print(
    "Probability range:",
    f"{smoke_probs.min():.6f}",
    "to",
    f"{smoke_probs.max():.6f}"
)

Positive samples: 194
Negative samples: 1854
Expected pos_weight: 9.556701030927835
Adapter pos_weight: 9.556701030927835
Class weighting: fold_local_positive_class_weight

Weighted FT-Transformer smoke test PASSED.
Probability range: 0.471666 to 0.514706


In [5]:
# -----------------------------
# Preserve previous unweighted FT-Transformer results
# -----------------------------

from pathlib import Path
import shutil


RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

UNWEIGHTED_DIR = (
    RESULTS_DIR
    / "unweighted_baseline"
)

UNWEIGHTED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Files produced by the previous unweighted experiment.
unweighted_artifacts = [
    "figure_01_ft_transformer_precision_recall_curve.png",
    "figure_02_ft_transformer_oof_probability_distribution.png",
    "ft_transformer_fold_metrics.csv",
    "ft_transformer_oof_predictions.csv",
    "ft_transformer_output_manifest.csv",
    "ft_transformer_run_metadata.json",
    "ft_transformer_summary_metrics.json",
]


moved_files = []
missing_files = []


for filename in unweighted_artifacts:

    source = RESULTS_DIR / filename
    destination = UNWEIGHTED_DIR / filename

    if source.exists():

        shutil.move(
            str(source),
            str(destination),
        )

        moved_files.append(filename)

    else:
        missing_files.append(filename)


print("Unweighted baseline archive:")
print("Moved:", len(moved_files))

for filename in moved_files:
    print("  ✓", filename)


if missing_files:

    print("\nNot found:")

    for filename in missing_files:
        print("  -", filename)


print(
    "\nArchive directory:",
    UNWEIGHTED_DIR
)

Unweighted baseline archive:
Moved: 7
  ✓ figure_01_ft_transformer_precision_recall_curve.png
  ✓ figure_02_ft_transformer_oof_probability_distribution.png
  ✓ ft_transformer_fold_metrics.csv
  ✓ ft_transformer_oof_predictions.csv
  ✓ ft_transformer_output_manifest.csv
  ✓ ft_transformer_run_metadata.json
  ✓ ft_transformer_summary_metrics.json

Archive directory: D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\unweighted_baseline


In [6]:
# -----------------------------
# Official weighted FT-Transformer 5-fold CV
# -----------------------------

import time

from src.models.runner import run_5fold_cv


# --------------------------------------------------
# Factory: fresh weighted adapter for every fold
# --------------------------------------------------

def make_weighted_ft_transformer():
    """
    Create a fresh FT-Transformer adapter.

    Class weighting is calculated automatically
    inside fit() using ONLY that fold's training labels.
    """

    return FTTransformerAdapter(
        name=model_name,
        model_id=model_id,
        params=fixed_params.copy(),
        categorical_features=categorical_feature_names.copy(),
    )


# --------------------------------------------------
# Development-only modeling data
# --------------------------------------------------

development_ids = set(
    split_registry.loc[
        split_registry["partition"] == "development",
        "Employee_ID",
    ]
)

holdout_ids = set(
    split_registry.loc[
        split_registry["partition"] == "holdout",
        "Employee_ID",
    ]
)

development_data = (
    df.loc[
        df["Employee_ID"].isin(development_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


# --------------------------------------------------
# Final leakage checks
# --------------------------------------------------

assert len(development_data) == 80_000

assert development_data[
    "Employee_ID"
].is_unique

assert set(
    development_data["Employee_ID"]
).isdisjoint(
    holdout_ids
), "Formal holdout leakage detected."


# --------------------------------------------------
# Environment summary
# --------------------------------------------------

device_name = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("Weighted FT-Transformer CV")
print("--------------------------")
print("Development rows:", len(development_data))
print("Formal holdout rows used: 0")
print("Predictors:", len(primary_model_features))
print("Outer folds:", 5)
print("Training rows per fold:", 64_000)
print("Validation rows per fold:", 16_000)
print("Epochs per fold:", 50)
print("Batch size:", 512)
print("Device:", device_name)
print("Class weighting: fold-local pos_weight")

print(
    "\nEach fold calculates:"
    "\npos_weight = negative training samples / positive training samples"
)

print("\nStarting weighted 5-fold training...")


# --------------------------------------------------
# Run weighted leakage-safe CV
# --------------------------------------------------

weighted_cv_start = time.perf_counter()


weighted_ft_cv_result = run_5fold_cv(
    adapter_factory=make_weighted_ft_transformer,

    # Development data ONLY
    data=development_data,

    # Frozen split registry
    split_registry=split_registry,

    # Approved 17 predictors
    feature_cols=primary_model_features,

    target_col=target_column,
    id_col="Employee_ID",

    fold_col="cv_fold",
    partition_col="partition",

    development_label="development",
    holdout_label="holdout",

    expected_folds=(1, 2, 3, 4, 5),

    expected_model_name=model_name,
)


weighted_cv_elapsed = (
    time.perf_counter()
    - weighted_cv_start
)


# --------------------------------------------------
# Completion
# --------------------------------------------------

print(
    "\nWeighted FT-Transformer "
    "5-fold CV completed successfully."
)

print(
    "Total elapsed time:",
    f"{weighted_cv_elapsed / 60:.2f} minutes"
)

print(
    "Result object:",
    type(weighted_ft_cv_result).__name__
)

print(
    "Final-fold pos_weight:",
    weighted_ft_cv_result
    .metadata
    .get("pos_weight")
)

Weighted FT-Transformer CV
--------------------------
Development rows: 80000
Formal holdout rows used: 0
Predictors: 17
Outer folds: 5
Training rows per fold: 64000
Validation rows per fold: 16000
Epochs per fold: 50
Batch size: 512
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Class weighting: fold-local pos_weight

Each fold calculates:
pos_weight = negative training samples / positive training samples

Starting weighted 5-fold training...

Weighted FT-Transformer 5-fold CV completed successfully.
Total elapsed time: 16.12 minutes
Result object: OOFResult
Final-fold pos_weight: 8.990633780830471


In [7]:
# -----------------------------
# Inspect weighted FT-Transformer results
# -----------------------------

print("Weighted FT-Transformer — pooled metrics")
print("----------------------------------------")

summary = weighted_ft_cv_result.summary_metrics

print(
    "PR-AUC:",
    f"{summary['pooled_average_precision']:.6f}",
    f"({summary['pooled_average_precision'] * 100:.2f}%)"
)

print(
    "ROC-AUC:",
    f"{summary['pooled_roc_auc']:.6f}",
    f"({summary['pooled_roc_auc'] * 100:.2f}%)"
)

print(
    "F1:",
    f"{summary['pooled_f1']:.6f}",
    f"({summary['pooled_f1'] * 100:.2f}%)"
)

print(
    "Brier Score:",
    f"{summary['pooled_brier_score']:.6f}"
)


print("\n5-fold mean ± SD")
print("----------------")

print(
    "PR-AUC:",
    f"{summary['cv_mean_average_precision']:.6f}",
    "±",
    f"{summary['cv_std_average_precision']:.6f}"
)

print(
    "ROC-AUC:",
    f"{summary['cv_mean_roc_auc']:.6f}",
    "±",
    f"{summary['cv_std_roc_auc']:.6f}"
)

print(
    "F1:",
    f"{summary['cv_mean_f1']:.6f}",
    "±",
    f"{summary['cv_std_f1']:.6f}"
)


print("\nFold-level metrics:")
display(
    weighted_ft_cv_result.fold_metrics
)

Weighted FT-Transformer — pooled metrics
----------------------------------------
PR-AUC: 0.098820 (9.88%)
ROC-AUC: 0.495721 (49.57%)
F1: 0.166441 (16.64%)
Brier Score: 0.251497

5-fold mean ± SD
----------------
PR-AUC: 0.098917 ± 0.002297
ROC-AUC: 0.494650 ± 0.008033
F1: 0.165884 ± 0.005804

Fold-level metrics:


,average_precision,roc_auc,precision,recall,f1,balanced_accuracy,accuracy,brier_score,tn,fp,fn,tp,predicted_positive_count,predicted_positive_rate,fold,fit_time_seconds,predict_time_seconds
0,0.097119,0.489921,0.096327,0.434104,0.157668,0.490647,0.535875,0.247108,7879,6520,906,695,7215,0.450937,1,91.588546,0.422052
1,0.101821,0.504905,0.101658,0.409744,0.162900,0.503573,0.578625,0.247860,8602,5797,945,656,6453,0.403313,2,160.785029,0.231031
2,0.096267,0.483871,0.097312,0.583021,0.166786,0.490635,0.416750,0.255135,5734,8664,668,934,9598,0.599875,3,244.084570,0.251736
3,0.100469,0.496719,0.098975,0.590512,0.169534,0.496187,0.420750,0.250693,5786,8612,656,946,9558,0.597375,4,313.831654,0.222035
4,0.098908,0.497833,0.100844,0.596754,0.172532,0.502364,0.426875,0.256691,5874,8524,646,956,9480,0.592500,5,154.782610,0.455462


## FT-Transformer Experiment Progress and Class-Imbalance Handling

The FT-Transformer experiment was developed and evaluated using the project's frozen 5-fold cross-validation setup on the 80,000-row development partition. The 20,000-row formal holdout set remained excluded throughout model development.

### 1. Initial FT-Transformer Setup

The model used all 17 approved predictors:

- 13 numerical / ordinal features
- 4 categorical features

Numerical features were standardized using `StandardScaler`, while categorical features were encoded independently inside each training fold.

A fresh FT-Transformer model and fresh preprocessing objects were created for every cross-validation fold to prevent information leakage.

The model used the following main architecture:

- 3 Transformer blocks
- Block dimension: 64
- 4 attention heads
- Feed-forward hidden dimension: 128
- Attention dropout: 0.1
- Feed-forward dropout: 0.1
- AdamW optimizer
- 50 training epochs
- Batch size: 512
- Random seed: 42

During implementation, compatibility with the installed `rtdl_revisiting_models` version was also corrected by explicitly supplying:

`ffn_d_hidden_multiplier=None`

The final training environment used an NVIDIA GeForce RTX 4060 Laptop GPU through CUDA-enabled PyTorch.

---

### 2. Initial Unweighted FT-Transformer Experiment

The first FT-Transformer experiment used the standard binary cross-entropy loss:

`BCEWithLogitsLoss()`

No class weighting was applied.

However, the resignation target was strongly imbalanced:

- Approximately 89.99% of employees did not resign.
- Approximately 10.01% of employees resigned.

The initial unweighted model produced the following pooled out-of-fold results:

| Metric | Result |
|---|---:|
| PR-AUC | 10.11% |
| ROC-AUC | 49.88% |
| F1-score at threshold 0.5 | 0.00% |
| Brier Score | 0.0907 |

The F1-score was 0 because the model assigned low resignation probabilities to almost all employees. At the standard classification threshold of 0.5, no useful positive resignation predictions were produced, resulting in zero recall and therefore an F1-score of zero.

More importantly, PR-AUC and ROC-AUC were also close to no-skill performance. Since the resignation prevalence was approximately 10%, a PR-AUC of approximately 10% indicated that the model had very little ability to rank actual resigners above non-resigners.

---

### 3. Introducing Fold-Local Class Weighting

Because the project requires class-imbalance handling, the FT-Transformer training loss was updated to use a positive-class weight.

For each training fold, the positive-class weight was calculated independently as:

`pos_weight = number of negative training samples / number of positive training samples`

The weight was calculated only from the current outer training fold. The validation fold and formal holdout were never used when calculating the class weight.

The weighted loss therefore became:

`BCEWithLogitsLoss(pos_weight=pos_weight_tensor)`

With the approximately 90/10 class distribution, the calculated positive-class weight was close to 9 for each fold.

For example, the final fold produced:

`pos_weight ≈ 8.99`

This means that incorrectly handling an employee who actually resigned contributed substantially more to the training loss than incorrectly handling a non-resigned employee.

A small weighted smoke test was performed before the full experiment. It confirmed that the fold-local weight was calculated correctly and that weighted FT-Transformer training and prediction worked successfully.

---

### 4. Weighted FT-Transformer Experiment

The complete weighted FT-Transformer was then evaluated again using the same frozen 5-fold development CV structure.

Each fold contained:

- 64,000 training employees
- 16,000 validation employees

The full weighted experiment completed successfully on the RTX 4060 GPU in approximately 16.12 minutes.

The weighted model produced:

| Metric | Unweighted | Weighted |
|---|---:|---:|
| PR-AUC | 10.11% | 9.88% |
| ROC-AUC | 49.88% | 49.57% |
| F1-score | 0.00% | 16.64% |
| Brier Score | 0.0907 | 0.2515 |

The weighted 5-fold mean results were:

- PR-AUC: 9.89% ± 0.23%
- ROC-AUC: 49.47% ± 0.80%
- F1-score: 16.59% ± 0.58%

---

### 5. Interpretation

Class weighting clearly changed the model's classification behaviour.

The unweighted model assigned low resignation probabilities to almost everyone, causing the default 0.5 threshold to produce an F1-score of zero.

After applying class weighting, the model became much more willing to classify employees as potential resigners. As a result:

- Recall increased substantially.
- F1-score increased from 0% to approximately 16.64%.
- The model detected many more actual resignation cases.

However, this improvement came with a large number of false-positive predictions.

Across the folds, precision remained close to approximately 10%, while recall was approximately 41% to 60%. Therefore, although the weighted model detected more resigners, many employees predicted as likely to resign were actually non-resigners.

More importantly, class weighting did not improve the model's ranking performance:

- Weighted PR-AUC remained approximately equal to the 10% resignation prevalence.
- Weighted ROC-AUC remained approximately 50%.
- Balanced accuracy also remained close to 50%.

The Brier Score also increased from approximately 0.091 to 0.251, showing that the weighted model's raw probabilities became less well calibrated.

---

### 6. Current Conclusion

Class imbalance was an important reason why the initial FT-Transformer produced an F1-score of zero at the default 0.5 threshold.

Applying fold-local positive-class weighting successfully corrected this behaviour and substantially improved recall and F1-score.

However, class weighting did not improve PR-AUC or ROC-AUC. Both remained approximately at no-skill levels.

Therefore, the results suggest that class imbalance alone is not the main limitation of the current FT-Transformer model. The available predictors may contain limited information for distinguishing employees who resign from employees who remain, or the current model configuration may require further tuning.

The weighted FT-Transformer remains methodologically important because it satisfies the class-imbalance requirement and demonstrates the effect of weighting. The unweighted experiment is retained as a baseline for comparison.

Further analysis should examine the out-of-fold probability distribution and threshold behaviour before making the final FT-Transformer assessment.

In [8]:
# -----------------------------
# Inspect weighted OOF predictions
# -----------------------------

weighted_oof = (
    weighted_ft_cv_result
    .predictions
    .copy()
)


print(
    "OOF prediction shape:",
    weighted_oof.shape
)

print(
    "\nColumns:"
)

print(
    weighted_oof.columns.tolist()
)


print(
    "\nFirst 5 rows:"
)

display(
    weighted_oof.head()
)


print(
    "\nMissing values per column:"
)

print(
    weighted_oof
    .isna()
    .sum()
)

OOF prediction shape: (80000, 6)

Columns:
['Employee_ID', 'cv_fold', 'y_true', 'model_name', 'y_probability', 'y_pred_0_5']

First 5 rows:


,Employee_ID,cv_fold,y_true,model_name,y_probability,y_pred_0_5
0,1,3,0,FT-Transformer,0.498213,0
1,2,3,0,FT-Transformer,0.499968,0
2,4,5,0,FT-Transformer,0.492771,0
3,5,3,0,FT-Transformer,0.508392,1
4,7,3,0,FT-Transformer,0.502569,1



Missing values per column:
Employee_ID      0
cv_fold          0
y_true           0
model_name       0
y_probability    0
y_pred_0_5       0
dtype: int64


In [9]:
# -----------------------------
# Weighted FT-Transformer threshold analysis
# -----------------------------

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
)


# Actual labels and OOF probabilities.
y_true_oof = (
    weighted_oof["y_true"]
    .astype(int)
    .to_numpy()
)

y_prob_oof = (
    weighted_oof["y_probability"]
    .astype(float)
    .to_numpy()
)


# Test thresholds from 0.05 to 0.95.
thresholds = np.arange(
    0.05,
    0.951,
    0.01
)


threshold_results = []


for threshold in thresholds:

    y_pred = (
        y_prob_oof >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_oof,
        y_pred,
        labels=[0, 1]
    ).ravel()

    threshold_results.append(
        {
            "threshold": threshold,

            "precision": precision_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "recall": recall_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "f1": f1_score(
                y_true_oof,
                y_pred,
                zero_division=0
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_true_oof,
                y_pred
            ),

            "accuracy": accuracy_score(
                y_true_oof,
                y_pred
            ),

            "predicted_positive_rate": y_pred.mean(),

            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "true_negatives": tn,
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


# --------------------------------------------------
# Find threshold with highest OOF F1
# --------------------------------------------------

best_f1_row = (
    threshold_df
    .loc[
        threshold_df["f1"].idxmax()
    ]
)


print("Best OOF threshold by F1")
print("------------------------")

print(
    "Threshold:",
    f"{best_f1_row['threshold']:.2f}"
)

print(
    "Precision:",
    f"{best_f1_row['precision'] * 100:.2f}%"
)

print(
    "Recall:",
    f"{best_f1_row['recall'] * 100:.2f}%"
)

print(
    "F1:",
    f"{best_f1_row['f1'] * 100:.2f}%"
)

print(
    "Balanced accuracy:",
    f"{best_f1_row['balanced_accuracy'] * 100:.2f}%"
)

print(
    "Predicted positive rate:",
    f"{best_f1_row['predicted_positive_rate'] * 100:.2f}%"
)


print("\nConfusion matrix counts:")

print(
    "TP:",
    int(best_f1_row["true_positives"])
)

print(
    "FP:",
    int(best_f1_row["false_positives"])
)

print(
    "FN:",
    int(best_f1_row["false_negatives"])
)

print(
    "TN:",
    int(best_f1_row["true_negatives"])
)


# --------------------------------------------------
# Compare useful thresholds around 0.50
# --------------------------------------------------

comparison_thresholds = [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
]


display(
    threshold_df[
        threshold_df["threshold"]
        .round(2)
        .isin(comparison_thresholds)
    ][
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "balanced_accuracy",
            "accuracy",
            "predicted_positive_rate",
        ]
    ]
)

Best OOF threshold by F1
------------------------
Threshold: 0.05
Precision: 10.01%
Recall: 100.00%
F1: 18.20%
Balanced accuracy: 50.00%
Predicted positive rate: 100.00%

Confusion matrix counts:
TP: 8008
FP: 71992
FN: 0
TN: 0


,threshold,precision,recall,f1,balanced_accuracy,accuracy,predicted_positive_rate
35,0.40,0.100061,0.983142,0.181636,0.499787,0.113200,0.983525
40,0.45,0.100085,0.908966,0.180316,0.499926,0.172775,0.909100
45,0.50,0.098974,0.522852,0.166441,0.496695,0.475775,0.528800
50,0.55,0.096268,0.062812,0.076022,0.498611,0.847163,0.065312
55,0.60,0.108200,0.011863,0.021382,0.500493,0.891300,0.010975


In [10]:
# -----------------------------
# Save weighted fixed FT-Transformer experiment
# -----------------------------

import json
from pathlib import Path


# --------------------------------------------------
# Output directory
# --------------------------------------------------

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "models"
    / "ft_transformer"
)

WEIGHTED_FIXED_DIR = (
    RESULTS_DIR
    / "weighted_fixed_baseline"
)

WEIGHTED_FIXED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# 1. Save OOF predictions
# --------------------------------------------------

weighted_oof_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_oof_predictions.csv"
)

weighted_ft_cv_result.predictions.to_csv(
    weighted_oof_path,
    index=False,
)


# --------------------------------------------------
# 2. Save fold-level metrics
# --------------------------------------------------

weighted_fold_metrics_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_fold_metrics.csv"
)

weighted_ft_cv_result.fold_metrics.to_csv(
    weighted_fold_metrics_path,
    index=False,
)


# --------------------------------------------------
# 3. Save pooled / summary metrics
# --------------------------------------------------

weighted_summary_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_summary_metrics.json"
)

with open(
    weighted_summary_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        weighted_ft_cv_result.summary_metrics,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# 4. Save threshold analysis
# --------------------------------------------------

threshold_analysis_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_threshold_analysis.csv"
)

threshold_df.to_csv(
    threshold_analysis_path,
    index=False,
)


# --------------------------------------------------
# 5. Save run metadata
# --------------------------------------------------

run_metadata = {
    "model": "FT-Transformer",
    "experiment": "weighted_fixed_baseline",

    "development_rows": 80_000,
    "formal_holdout_rows_used": 0,

    "features": 17,
    "cv_folds": 5,

    "epochs_per_fold": 50,
    "batch_size": 512,

    "class_weighting": "fold_local_pos_weight",
    "pos_weight_formula": "negative_count / positive_count",

    "final_fold_pos_weight": (
        weighted_ft_cv_result
        .metadata
        .get("pos_weight")
    ),

    "training_device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),

    "elapsed_minutes": weighted_cv_elapsed / 60,

    "primary_metric": "PR-AUC",

    "pooled_pr_auc": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_average_precision"
        ]
    ),

    "pooled_roc_auc": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_roc_auc"
        ]
    ),

    "pooled_f1_at_0_5": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_f1"
        ]
    ),

    "pooled_brier_score": (
        weighted_ft_cv_result
        .summary_metrics[
            "pooled_brier_score"
        ]
    ),

    "best_oof_f1_threshold": float(
        best_f1_row["threshold"]
    ),

    "best_oof_f1": float(
        best_f1_row["f1"]
    ),

    "best_threshold_predicted_positive_rate": float(
        best_f1_row[
            "predicted_positive_rate"
        ]
    ),

    "threshold_conclusion": (
        "The numerically best OOF F1 threshold predicts "
        "essentially/all development employees as positive, "
        "so it is not considered a useful decision threshold."
    ),
}


run_metadata_path = (
    WEIGHTED_FIXED_DIR
    / "ft_transformer_weighted_run_metadata.json"
)

with open(
    run_metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        run_metadata,
        file,
        indent=4,
        default=float,
    )


# --------------------------------------------------
# Verification
# --------------------------------------------------

saved_files = [
    weighted_oof_path,
    weighted_fold_metrics_path,
    weighted_summary_path,
    threshold_analysis_path,
    run_metadata_path,
]


print("Weighted fixed experiment saved.")
print("\nDirectory:")
print(WEIGHTED_FIXED_DIR)

print("\nSaved files:")

for path in saved_files:
    print(
        " ✓",
        path.name,
        f"({path.stat().st_size:,} bytes)"
    )


# Critical integrity checks
assert len(
    weighted_ft_cv_result.predictions
) == 80_000

assert (
    weighted_ft_cv_result
    .predictions["Employee_ID"]
    .is_unique
)

assert set(
    weighted_ft_cv_result
    .predictions["Employee_ID"]
).isdisjoint(
    holdout_ids
)

assert all(
    path.exists()
    for path in saved_files
)


print(
    "\nWeighted experiment artifact checks PASSED."
)

Weighted fixed experiment saved.

Directory:
D:\Desktop\Employee_Resignation_Prediction\results\models\ft_transformer\weighted_fixed_baseline

Saved files:
 ✓ ft_transformer_weighted_oof_predictions.csv (3,760,368 bytes)
 ✓ ft_transformer_weighted_fold_metrics.csv (1,282 bytes)
 ✓ ft_transformer_weighted_summary_metrics.json (708 bytes)
 ✓ ft_transformer_weighted_threshold_analysis.csv (9,251 bytes)
 ✓ ft_transformer_weighted_run_metadata.json (1,034 bytes)

Weighted experiment artifact checks PASSED.
